**Necessary imports**

In [22]:
import subprocess
import os
import pandas as pd
import json

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


**Setting parameters**

In [23]:
simple_render_raw_data_dir = "/dlabdata1/tsoares/wikidumps/"
root_processing_dir = "/dlabdata1/boulenge/wikidumps_data"
experiment_name = "selected_langs_small_sample_3"
processed_data_dir = os.path.join(root_processing_dir, experiment_name)
server_out_dir = os.path.join("/var/www/html/wikinsert", experiment_name)
processes = 48
min_nb_samples = 5
revisions_sampling_size = 0.0003 #10% of the revisions 

#For all possible languages
all_langs = sorted(set([filename.split("wiki")[0] for filename in os.listdir(simple_render_raw_data_dir) if "wiki" in filename]))

selected_langs_path = 'selected_languages.json'

with open(selected_langs_path, 'r') as f:
   langs = json.load(f)

side_context_sizes = [5]  #List of context size to try
second_month = "20231001"
third_month = "20231101"

langs_to_remove = ["war", "ky", "nds-nl"] #These languages do not work

for lang in langs_to_remove:
    if lang in all_langs:
        all_langs.remove(lang)


#langs = langs[:2] #For testing purposes

print("Will process languages:\n" + '\n'.join(langs))



Will process languages:
af
ar
cs
cy
de
en
es
fa
fr
ga
gu
he
hi
is
it
ja
jv
kk
kn
ms
pt
ru
simple
sk
sw
tr
ur
uz
vi
zh


**Defining data paths**

In [24]:
# Define directories
all_revisions_dir = os.path.join(root_processing_dir, "all_revisions")
rev_files_dir = os.path.join(processed_data_dir, "revisions_files")
parsed_htmls_dir = os.path.join(processed_data_dir, "parsed_htmls")
contexts_splitted_data_dir = os.path.join(processed_data_dir, "contexts_splitted_data")
ready_to_score_data_dir = os.path.join(processed_data_dir, "ready_to_score_data")
scored_data_dir = os.path.join(processed_data_dir, "scored_data")
#crawling_output_dir = os.path.join(processed_data_dir, "crawled_pages")
crawling_output_dir = os.path.join(server_out_dir, "crawled_pages")

scripts_dir = "scripts"

# Create necessary directories if they don't exist
os.makedirs(all_revisions_dir, exist_ok=True)
os.makedirs(rev_files_dir, exist_ok=True)
os.makedirs(parsed_htmls_dir, exist_ok=True)
os.makedirs(contexts_splitted_data_dir, exist_ok=True)
os.makedirs(ready_to_score_data_dir, exist_ok=True)
os.makedirs(scored_data_dir, exist_ok=True)
os.makedirs(crawling_output_dir, exist_ok=True)
os.makedirs(server_out_dir, exist_ok=True)

**Prepare files to be crawled and parsed**

**Carefull, the following is only needed to be ran once for all (common for any experiment), so was probably ran before**

It generates the diffenret pages ids we will crawl, using data from the previous study

In [25]:
"""for lang in all_langs:
        print(f"Processing language: {lang}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{all_revisions_dir}/revisions_{lang}.parquet"
        if not os.path.exists(save_path):
            print("Generating revisions file")    
            subprocess.run([
                "python", f"{scripts_dir}/generate_revisions_file.py",
                "--versions_file", f"{second_month_path}/eval/link_versions",
                "--html_dir", f"{second_month_path}/eval/pages",
                "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
                "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
                "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
                "--save_path", save_path
            ])
        else:
            print("Revisions file already exists")"""


'for lang in all_langs:\n        print(f"Processing language: {lang}")\n        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"\n        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"\n        save_path = f"{all_revisions_dir}/revisions_{lang}.parquet"\n        if not os.path.exists(save_path):\n            print("Generating revisions file")    \n            subprocess.run([\n                "python", "generate_revisions_file.py",\n                "--versions_file", f"{second_month_path}/eval/link_versions",\n                "--html_dir", f"{second_month_path}/eval/pages",\n                "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",\n                "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",\n                "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",\n                "--save_path", save_path\n            ])\n        else:\n 

**Sampling**

In [26]:
langs_stats = []

for lang in langs:
    full_revisions_path = f"{all_revisions_dir}/revisions_{lang}.parquet"
    full_revisions_df = pd.read_parquet(full_revisions_path)
    print(f"Sampling language: {lang}")
    nb_total = len(full_revisions_df)
    pre_min_to_sample = 2*min_nb_samples #we take twice the number to prevent future loss of data
    if nb_total > pre_min_to_sample:
        pre_nb_samples = max(pre_min_to_sample, int(revisions_sampling_size*2*nb_total))
    else:
        pre_nb_samples = nb_total

    if nb_total > min_nb_samples:
        nb_samples = max(min_nb_samples, int(revisions_sampling_size*nb_total))
    else:
        nb_samples = nb_total

    langs_stats.append({
        "lang": lang,
        "orig_nb_revisions": nb_total,
        "pre_nb_samples": pre_nb_samples,
        "nb_samples": nb_samples
    })

    sampled_revisions_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    sampled_revisions = full_revisions_df.sample(n=pre_nb_samples, random_state=42)
    sampled_revisions.to_parquet(sampled_revisions_path)

display(pd.DataFrame(langs_stats))
display(pd.DataFrame(langs_stats).sum())


Sampling language: af


Sampling language: ar
Sampling language: cs
Sampling language: cy
Sampling language: de
Sampling language: en
Sampling language: es
Sampling language: fa
Sampling language: fr
Sampling language: ga
Sampling language: gu
Sampling language: he
Sampling language: hi
Sampling language: is
Sampling language: it
Sampling language: ja
Sampling language: jv
Sampling language: kk
Sampling language: kn
Sampling language: ms
Sampling language: pt
Sampling language: ru
Sampling language: simple
Sampling language: sk
Sampling language: sw
Sampling language: tr
Sampling language: ur
Sampling language: uz
Sampling language: vi
Sampling language: zh


,lang,orig_nb_revisions,pre_nb_samples,nb_samples
0,af,707,10,5
1,ar,6700,10,5
2,cs,4391,10,5
3,cy,238,10,5
4,de,49899,29,14
5,en,194300,116,58
6,es,29970,17,8
7,fa,7967,10,5
8,fr,26847,16,8
9,ga,109,10,5


lang                 afarcscydeenesfafrgaguhehiisitjajvkkknmsptrusi...
orig_nb_revisions                                               467883
pre_nb_samples                                                     454
nb_samples                                                         225
dtype: object

**Crawling first versions only**

There is no need to re-crawl the second version with all its files as the index is sufficent for these.

In [27]:
for lang in langs:
    print(f"Processing language: {lang}")
    subprocess.run([
        "python", f"{scripts_dir}/crawl.py",
        "--revisions_path", f"{rev_files_dir}/revisions_{lang}.parquet",
        "--output_dir", f"{crawling_output_dir}/{lang}",
        "--n_processes", str(processes),
        "--lang", lang
    ])

Processing language: af


Total number of pages to crawl: 10
Running crawling on 48 processes
Crawling finished in 0.06668637990951538 minutes, post processing...


Moving files: 100%|██████████| 10/10 [00:00<00:00, 6760.64it/s]


Processing language: ar
Total number of pages to crawl: 10
Running crawling on 48 processes
Crawling finished in 0.16956610282262166 minutes, post processing...


Moving files: 100%|██████████| 10/10 [00:00<00:00, 1828.78it/s]


...

Moving files: 100%|██████████| 10/10 [00:00<00:00, 4829.37it/s]


Processing language: vi
Total number of pages to crawl: 10
Running crawling on 48 processes
Crawling finished in 0.1813191254933675 minutes, post processing...


Moving files: 100%|██████████| 10/10 [00:00<00:00, 3077.71it/s]


Processing language: zh
Total number of pages to crawl: 10
Running crawling on 48 processes
Crawling finished in 0.149397873878479 minutes, post processing...


Moving files: 100%|██████████| 10/10 [00:00<00:00, 8586.09it/s]


**Parsing of the htmls and sentence tokenizing**

In [28]:
for lang in langs:
    print(f"Processing language: {lang}")
    second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
    first_version_specific_html_dir = f"{crawling_output_dir}/{lang}"
    save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    output_dir = f"{parsed_htmls_dir}/{lang}_v2"
    first_version_specific_output_dir = f"{parsed_htmls_dir}/{lang}_v1"
        
    subprocess.run([
        "python", f"{scripts_dir}/parsing.py",
        "--revisions_path", save_path,        
        "--html_dir", f"{second_month_path}/eval/pages",
        "--first_versions_specific_html_dir", first_version_specific_html_dir,
        "--output_dir", output_dir,
        "--first_versions_specific_output_dir", first_version_specific_output_dir,
        "--lang", lang,
        "--n_processes", str(processes)
    ])

    parsed_htmls_paths = os.listdir(first_version_specific_output_dir)
    sentences_path = list(
        filter(
            lambda x: (
                x.endswith("sentences.jsonl") and ("raw" not in x)
                ),
            parsed_htmls_paths
            )
        )[0]
    links_path = list(filter(lambda x: x.endswith("links.jsonl"), parsed_htmls_paths))[0]
    sentences_df = pd.read_json(f"{first_version_specific_output_dir}/{sentences_path}")
    links_df = pd.read_json(f"{first_version_specific_output_dir}/{links_path}")

    display(sentences_df.head(5))
    display(links_df.head(5)) 

Processing language: af
First version html dir: /var/www/html/wikinsert/selected_langs_small_sample_3/crawled_pages/af
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1


Sentence tokenization: 100%|██████████| 10/10 [00:00<00:00, 41.77it/s]


Time taken for v1: 10.453876972198486
Writing sentences and links v2


Sentence tokenization: 100%|██████████| 10/10 [00:00<00:00, 39.72it/s]


,md5,id,text,startIdx,endIdx,section,locList
0,1509afbe912fc0e70ec3d7a083f7e21b,0,Piket-Bo-Berg is 'n plaasgemeenskap geleë in d...,34245,34372,Lead,"[{'start': 34245, 'end': 34258}, {'start': 342..."
1,c771bb69375660fb63a2c352317aafc7,1,Die gemeenskap lê bo-op die Piketberge en is n...,34372,34523,Lead,"[{'start': 34372, 'end': 34440}, {'start': 344..."
2,47b40cd8340f99c7be3ce1ea8b867e63,2,Dit is oor tyd deur 'n verskeidenheid boere be...,34523,34635,Lead,"[{'start': 34523, 'end': 34635}]"
3,49a811f19c48f823bf7f53ecdee39c16,3,"Voorheen is die Bo-Berg bewoon deur Boesmanne,...",34635,34729,Lead,"[{'start': 34635, 'end': 34729}]"
4,166bf21d3b5aa648f209a64e799810a6,4,In 1810 was daar vyf plase wat daar op die Ber...,34729,34797,Lead,"[{'start': 34736, 'end': 34797}]"


...

Sentence tokenization: 100%|██████████| 10/10 [00:00<00:00, 70.19it/s]


Time taken for v1: 28.62204122543335
Writing sentences and links v2


Sentence tokenization: 100%|██████████| 10/10 [00:00<00:00, 74.76it/s]


,md5,id,text,startIdx,endIdx,section,locList
0,a87b7f9939b52e11c45b15832a61fbe3,0,"\n未入流是明、清兩代對於低於從九品的官吏的稱呼，在唐代則被稱為""流外""。",45736,45842,Lead,"[{'start': 45735, 'end': 45736}, {'start': 457..."
1,4f2ce8202fa1e68690e4e85d6d10612d,1,\n\n\n清代:\n文官\n,45842,46104,Lead,"[{'start': 45846, 'end': 45847}, {'start': 458..."
2,1573eef75b4096959a9ecb691587fbd3,2,武官\n,56576,56587,文武官職,"[{'start': 56575, 'end': 56576}, {'start': 565..."
3,9a8d45681362a72287918ad2de8cede3,3,《明史．卷七二．職官志一》：「凡文官之品九，品有正從，為級一十八。,58254,58290,文獻提及,"[{'start': 58253, 'end': 58254}, {'start': 582..."
4,decd6e782d8676fe47b96b8b6eb645cf,4,不及九品曰未入流。,58290,58299,文獻提及,"[{'start': 58290, 'end': 58299}]"


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/%E7%83%8F%E5%85%8B%E8%98%AD%E8%AA%9E,46591,烏克蘭語,46685,Lead,0
1,file:///wiki/%E7%83%8F%E5%85%8B%E8%98%AD,46719,烏克蘭,46802,Lead,0
2,file:///wiki/%E5%88%87%E7%88%BE%E5%B0%BC%E6%88...,46812,切爾尼戈夫州,46908,Lead,0
3,file:///wiki/%E4%BC%8A%E5%A5%87%E5%B0%BC%E4%BA...,46910,伊奇尼亞區,46995,Lead,0
4,file:///wiki/%E4%BA%92%E8%81%94%E7%BD%91%E6%A1...,47502,互联网档案馆,47598,參考資料,1


**Preprocessing**

It included all preprocessing parts but the merging

In [29]:
for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
        output_dir = f"{contexts_splitted_data_dir}/{lang}/context_size_{side_context_size}"
        
        subprocess.run([
            "python", f"{scripts_dir}/preprocessing.py",
            "--revisions_path", save_path,
            "--parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v2",
            "--first_versions_specific_parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v1",
            "--n_processes", str(processes),
            "--output_dir", output_dir,
            "--side_contexts_size", str(side_context_size),
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet"
        ])

        #preprocessed_data_path = os.path.join(output_dir, f'test_data.parquet')

        #preprocessed_data = pd.read_parquet(preprocessed_data_path)
        #display(preprocessed_data.head(5))

Processing language: af with side context size 5
Running on 48 processes


100%|██████████| 42/42 [00:00<00:00, 8427.13it/s]


The length of the output is 42
Processing language: ar with side context size 5
Running on 48 processes


100%|██████████| 10/10 [00:08<00:00,  1.17it/s]


The length of the output is 16


...

100%|██████████| 48/48 [00:00<00:00, 8876.83it/s]


Processing language: zh with side context size 5
Running on 48 processes


100%|██████████| 10/10 [00:08<00:00,  1.15it/s]


The length of the output is 20


100%|██████████| 20/20 [00:00<00:00, 10042.63it/s]


**Merging**

In [30]:
test_data_dir = contexts_splitted_data_dir
output_dir = ready_to_score_data_dir

subprocess.run([
    "python", f"{scripts_dir}/merge_data.py",
    "--test_data_dir", test_data_dir,
    "--raw_data_dir", simple_render_raw_data_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--output_dir", output_dir,
    "--ignore_warnings"
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
scored_files_example = os.listdir(output_dir)
scored_files_example = list(filter(lambda x: (first_lang in x and first_context_size in x), scored_files_example))
final_input_data_path = os.path.join(output_dir, scored_files_example[0])

final_input_data_df = pd.read_parquet(final_input_data_path)
display(final_input_data_df.head(5))

Saving mention maps:
Processing language af (1/30)
Finished processing language af
Processing language ar (2/30)
Finished processing language ar
Processing language cs (3/30)
Finished processing language cs
Processing language cy (4/30)
Finished processing language cy
Processing language de (5/30)
Finished processing language de
Processing language en (6/30)
Finished processing language en
Processing language es (7/30)
Finished processing language es
Processing language fa (8/30)
Finished processing language fa
Processing language fr (9/30)
Finished processing language fr
Processing language ga (10/30)
Finished processing language ga
Processing language gu (11/30)
Finished processing language gu
Processing language he (12/30)
Finished processing language he
Processing language hi (13/30)
Finished processing language hi
Processing language is (14/30)
Finished processing language is
Processing language it (15/30)
Finished processing language it
Processing language ja (16/30)
Finished pro

100%|██████████| 110890/110890 [00:00<00:00, 728902.88it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 42 links for language af
Removed 0 links because of empty contexts
Removed 13 links because target title is not in source titles
Removed 0 links because of missing section
After cleaning, there are 29 links for language af
Finished processing language af
Processing language ar (2/30)
Loaded pages
Loaded redirects


100%|██████████| 1209639/1209639 [00:01<00:00, 855963.38it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 16 links for language ar
Removed 0 links because of empty contexts
Removed 4 links because target title is not in source titles
Removed 1 links because of missing section
After cleaning, there are 11 links for language ar
Finished processing language ar
Processing language cs (3/30)
Loaded pages
Loaded redirects


...

100%|██████████| 1273459/1273459 [00:01<00:00, 866354.41it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 48 links for language vi
Removed 0 links because of empty contexts
Removed 6 links because target title is not in source titles
Removed 16 links because of missing section
After cleaning, there are 26 links for language vi
Finished processing language vi
Processing language zh (30/30)
Loaded pages
Loaded redirects


100%|██████████| 1344667/1344667 [00:01<00:00, 827062.92it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 20 links for language zh
Removed 0 links because of empty contexts
Removed 1 links because target title is not in source titles
Removed 0 links because of missing section
After cleaning, there are 19 links for language zh
Finished processing language zh


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts,target_lead
0,1942,486,9_Julie,None,[62],None,"9 Julie - Richard Roundtree, Amerikaanse akteu...",9 Julie,Geboortes,None,Die jaar 1942 was 'n gewone jaar wat volgens d...,2622176,2624839,None,missing_sentence,[{'context': 'Die jaar 1942 was 'n gewone jaar...,9 Julie is die 190ste dag van die jaar in die ...
1,Marius_Petipa,407068,Pjotr_Tsjaikofski,None,[167],None,Gedurende die goue era van die Russiese Ryk he...,Tsjaikofski,"Sint Petersburg, Rusland",None,Marius Ivanovich Petipa (Russies: Мариус Ивано...,2621414,2621682,None,missing_span,[{'context': 'Marius Ivanovich Petipa (Russies...,"Pjotr Iljitsj Tsjaikofski, Wotkinsk, 7 Mei 184..."
2,Marius_Petipa,407068,Groothertog_Aleksander_Michailowitsj_van_Rusland,None,[167],None,Gedurende die goue era van die Russiese Ryk he...,Groothertog Aleksander Michailowitsj van Rusland,"Sint Petersburg, Rusland",None,Marius Ivanovich Petipa (Russies: Мариус Ивано...,2621414,2621682,None,missing_span,[{'context': 'Marius Ivanovich Petipa (Russies...,Hierdie naam is volgens die Oos-Slawiese gebru...
3,Marius_Petipa,407068,Nikolaas_II_van_Rusland,None,[167],None,Gedurende die goue era van die Russiese Ryk he...,Keiser Nikolaas II,"Sint Petersburg, Rusland",None,Marius Ivanovich Petipa (Russies: Мариус Ивано...,2621414,2621682,None,missing_span,[{'context': 'Marius Ivanovich Petipa (Russies...,Nikolaas II Aleksandrowitsj was die laaste kei...
4,Marius_Petipa,407068,Giuseppe_Verdi,None,[167],None,Gedurende die goue era van die Russiese Ryk he...,Verdi,"Sint Petersburg, Rusland",None,Marius Ivanovich Petipa (Russies: Мариус Ивано...,2621414,2621682,None,missing_span,[{'context': 'Marius Ivanovich Petipa (Russies...,Giuseppe Fortunino Francesco Verdi was 'n Ital...


**Scoring the contexts**

In [31]:
data_dir = ready_to_score_data_dir
output_dir = scored_data_dir

for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")

        subprocess.run([
            "python", f"{scripts_dir}/get_llm_scores.py",
            "--data_dir", data_dir,
            "--output_dir", output_dir,
            "--lang", lang,
            "--side_contexts_size", str(side_context_size),
            "--model_dir", "/dlabdata1/tsoares/models",
            "--model_prefix", "roberta_full",
            "--use_section_title",
            "--batch_size", "128",
            "--use_mentions",
            "--use_cuda",
            "--multilingual",
            "--multilingual_name", "multilingual-all"
        ])

scored_files_example = os.listdir(output_dir)[0]
scored_data = pd.read_parquet(f"{output_dir}/{scored_files_example}")

exemple = scored_data['all_contexts'][0][0]

for key in exemple.keys():
    print(f"{key}:\n{exemple[key]}\n")

Processing language: af with side context size 5
Processing context size 5


100%|██████████| 40/40 [03:06<00:00,  4.65s/it]


Processing language: ar with side context size 5
Processing context size 5


100%|██████████| 15/15 [01:05<00:00,  4.40s/it]


Processing language: cs with side context size 5
Processing context size 5


...

Processing language: vi with side context size 5
Processing context size 5


100%|██████████| 24/24 [01:43<00:00,  4.33s/it]


Processing language: zh with side context size 5
Processing context size 5


100%|██████████| 9/9 [00:39<00:00,  4.43s/it]


context:
Ma Thì Hồ là một xã thuộc huyện Mường Chà, tỉnh Điện Biên, Việt Nam. Xã Ma Thì Hồ có diện tích 139,9&nbsp;km², dân số năm 2006 là 3.087 người, mật độ dân số đạt 22 người/km². Người dân xã Ma Thì Hồ vẫn còn khá là nghèo. Do trình độ hiểu biết và dân trí còn thấp. Chủ yếu dựa vào nghề trồng trọt. Chiếm đa số dân hiện nay trong xã Ma Thì Hồ là người  Dân tộc (cộng đồng) Mông.

middle_sentence:
{'clean_sentence': 'Ma Thì Hồ là một xã thuộc huyện Mường Chà, tỉnh Điện Biên, Việt Nam.', 'clean_sentence2': None, 'endIdx1': 43272, 'endIdx2': 43708.0, 'ids1': array([0]), 'ids2': array([0]), 'index': 0, 'locList1': array([{'end': 42938, 'start': 42937}, {'end': 42953, 'start': 42944},
       {'end': 42965, 'start': 42957}, {'end': 43032, 'start': 43030},
       {'end': 43049, 'start': 43036}, {'end': 43120, 'start': 43111},
       {'end': 43131, 'start': 43124}, {'end': 43202, 'start': 43193},
       {'end': 43208, 'start': 43206}, {'end': 43266, 'start': 43258},
       {'end': 43272, 's

**Colouring the pages**

In [32]:
sorted_link_output_dir = os.path.join(server_out_dir, "colored_pages")
soft_links_dir = os.path.join(server_out_dir, "soft_links")

# Run the coloring script
subprocess.run([
    "python", f"{scripts_dir}/color_pages.py",
    "--scored_data_dir", scored_data_dir,
    "--html_dir", crawling_output_dir,
    "--sorted_links_output_dir", sorted_link_output_dir,
    #"--soft_links_dir", soft_links_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--extra_crawled_files_dir", crawling_output_dir,
    "--full_model_name", "roberta_full_multilingual-all"
])


Processing language af with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1


  0%|          | 0/29 [00:00<?, ?it/s]

Style: highlight


  0%|          | 0/11 [00:00<?, ?it/s]

Processing language ar with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


...

Processing language vi with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


  0%|          | 0/19 [00:00<?, ?it/s]

Processing language zh with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


100%|██████████| 19/19 [00:01<00:00, 12.10it/s]


CompletedProcess(args=['python', 'color_pages.py', '--scored_data_dir', '/dlabdata1/boulenge/wikidumps_data/selected_langs_small_sample_3/scored_data', '--html_dir', '/var/www/html/wikinsert/selected_langs_small_sample_3/crawled_pages', '--sorted_links_output_dir', '/var/www/html/wikinsert/selected_langs_small_sample_3/colored_pages', '--langs', 'af,ar,cs,cy,de,en,es,fa,fr,ga,gu,he,hi,is,it,ja,jv,kk,kn,ms,pt,ru,simple,sk,sw,tr,ur,uz,vi,zh', '--side_contexts_sizes', '5', '--extra_crawled_files_dir', '/var/www/html/wikinsert/selected_langs_small_sample_3/crawled_pages', '--full_model_name', 'roberta_full_multilingual-all'], returncode=0)